# Tuning the history models on each rung

Notebook 13 tunes every history family once, on rung H4, and refits that setting on all seven rungs. This notebook asks whether the later rungs (H5, the SNR of earlier packets; H6, the sensors; H7, the level and the geometry of the link) add something when the model is tuned on their own inputs.

* **Families**: LightGBM and XGBoost, the two boosted-tree families, which can use any interaction the added inputs offer.
* **Search**: the search of notebook 13 (Bayesian, 20 candidates, every tenth training row, the five expanding folds, the same space and seed), run on the inputs of each rung from H4 to H7. On rung H4 it repeats the search of notebook 13, which shows how much a repeated search alone moves the result.
* **Fit and score**: the chosen setting refitted as in notebook 13 (out of fold for folds 0-4, all training rows for the hold-out); hold-out RMSE against that of the setting tuned on H4, read from the residual files of notebook 13 and paired by day, with 95 % intervals from resampling days; the 99 % margin from the out-of-fold residuals.

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"       # XGBoost on the GPU
import json
import time
import numpy as np
import pandas as pd
from skopt import BayesSearchCV
from skopt.space import Real, Integer
import xgboost as xgb
import lightgbm as lgb
from margin_tools import shift_pooled, day_codes, boot_stat

RANDOM_STATE = 42

# The rows, folds and inputs of notebook 13
train = pd.read_csv("Data_Files/history_train.csv")
test = pd.read_csv("Data_Files/history_test.csv")
rungs = json.load(open("Data_Files/history_rungs.json"))
d = pd.concat([train, test], ignore_index=True)
d["t"] = pd.to_datetime(d["t"], utc=True)
d = d.sort_values("t").reset_index(drop=True)
fold = d["fold"].to_numpy()
H = {k.split()[0]: v for k, v in rungs.items()}
ok = (d["y"].notna() & d[H["H7"]].notna().all(axis=1)).to_numpy()
cal, ev = ok & (fold >= 0) & (fold < 9), ok & (fold == 9)
y = d["y"].to_numpy()
X_all = d[H["H7"]].copy()
X_all = X_all.fillna(X_all[ok & (fold < 9)].median())
sub = np.flatnonzero(ok & (fold < 9))[::10]
cv_sub = [(np.flatnonzero(fold[sub] < k), np.flatnonzero(fold[sub] == k)) for k in range(5)]
day = day_codes(d["t"][ev])
h4_setting = json.load(open("CV_Results/history_best_params.json"))
info = pd.read_csv("CV_Results/information_by_family.csv", index_col=[0, 1])
rmse = lambda e: float(np.sqrt(np.mean(np.square(e))))
print(f"rows: search sample {len(sub):,} | calibration (folds 0-4) {int(cal.sum()):,} | hold-out {int(ev.sum()):,}")

rows: search sample 212,668 | calibration (folds 0-4) 1,773,285 | hold-out 531,983


In [2]:
# The search spaces and estimators of notebook 13
SEARCH = {
    "LGBM": (lgb.LGBMRegressor(subsample_freq=1, random_state=RANDOM_STATE, n_jobs=4, verbose=-1),
             {"num_leaves": Integer(15, 255), "learning_rate": Real(0.01, 0.2, prior="log-uniform"), "n_estimators": Integer(200, 1500),
              "min_child_samples": Integer(20, 1000), "subsample": Real(0.6, 1.0), "colsample_bytree": Real(0.5, 1.0), "reg_lambda": Real(0.1, 30, prior="log-uniform")}, 8),
    "XGB": (xgb.XGBRegressor(device="cuda", tree_method="hist", random_state=RANDOM_STATE, n_jobs=1),
            {"max_depth": Integer(3, 10), "learning_rate": Real(0.01, 0.2, prior="log-uniform"), "n_estimators": Integer(200, 1500),
             "min_child_weight": Real(1, 2000, prior="log-uniform"), "subsample": Real(0.6, 1.0), "colsample_bytree": Real(0.5, 1.0),
             "reg_lambda": Real(0.1, 30, prior="log-uniform")}, 1)}
FIT = {"LGBM": lambda p: lgb.LGBMRegressor(subsample_freq=1, random_state=RANDOM_STATE, n_jobs=8, verbose=-1, **p),
       "XGB": lambda p: xgb.XGBRegressor(device="cuda", tree_method="hist", random_state=RANDOM_STATE, **p)}

def tuned_on(fam, rung):
    """Search on the rung's inputs, refit the chosen setting as in notebook 13, and compare its hold-out residuals with those of the setting tuned on H4."""
    t0 = time.time()
    est, space, jobs = SEARCH[fam]
    X = X_all[H[rung]].to_numpy()
    bs = BayesSearchCV(est, space, n_iter=20, cv=cv_sub, scoring="neg_root_mean_squared_error", n_jobs=jobs, n_points=2, random_state=RANDOM_STATE, refit=False)
    bs.fit(X[sub], y[sub])
    setting = dict(bs.best_params_)
    p = np.full(len(d), np.nan)
    for k in range(5):
        tr, va = ok & (fold < k), ok & (fold == k)
        p[va] = FIT[fam](setting).fit(X[tr], y[tr]).predict(X[va])
    tr = ok & (fold < 9)
    p[ev] = FIT[fam](setting).fit(X[tr], y[tr]).predict(X[ev])
    e = y - p
    h4 = pd.read_csv(f"Residuals/residuals_{fam}_{rung}_test.csv", usecols=["row_id", "resid_db"])
    assert (h4["row_id"].to_numpy() == np.flatnonzero(ev)).all()
    e_h4 = h4["resid_db"].to_numpy()
    diff, lo, hi = boot_stat(day, np.column_stack([e[ev] ** 2, e_h4 ** 2, np.ones(ev.sum())]), lambda T: np.sqrt(T[:, 0] / T[:, 2]) - np.sqrt(T[:, 1] / T[:, 2]))
    margin = shift_pooled(e[cal], 0.99)
    return {"family": fam, "rung": rung, "setting": json.dumps({k: round(float(v), 4) for k, v in setting.items()}),
            "the H4 setting": all(np.isclose(float(setting[k]), float(h4_setting[fam][k])) for k in h4_setting[fam]),
            "sample CV RMSE": -bs.best_score_, "hold-out RMSE, H4 setting": rmse(e_h4), "hold-out RMSE, own setting": rmse(e[ev]),
            "difference": diff, "lo": lo, "hi": hi, "margin 0.99, H4 setting": info.loc[(fam, rung), "margin 0.99"], "margin 0.99, own setting": margin,
            "reached 0.99, own setting": float((e[ev] <= margin).mean()), "minutes": (time.time() - t0) / 60}

In [3]:
rows = []
for fam in ("LGBM", "XGB"):
    for rung in ("H4", "H5", "H6", "H7"):
        rows.append(tuned_on(fam, rung))
        r = rows[-1]
        print(f"{fam} {rung}: {'the H4 setting' if r['the H4 setting'] else 'another setting'}, hold-out RMSE {r['hold-out RMSE, own setting']:.4f} dB "
              f"against {r['hold-out RMSE, H4 setting']:.4f} with the H4 setting ({r['minutes']:.0f} min)")
retuning = pd.DataFrame(rows).set_index(["family", "rung"])
retuning.to_csv("CV_Results/history_retuning.csv")
pd.set_option("display.max_colwidth", 60)
display(retuning.drop(columns=["setting", "minutes"]).round(4))
display(retuning[["setting"]])

LGBM H4: the H4 setting, hold-out RMSE 1.7696 dB against 1.7696 with the H4 setting (5 min)


LGBM H5: the H4 setting, hold-out RMSE 1.7655 dB against 1.7655 with the H4 setting (5 min)


LGBM H6: another setting, hold-out RMSE 1.7687 dB against 1.7670 with the H4 setting (6 min)


LGBM H7: another setting, hold-out RMSE 1.7684 dB against 1.7672 with the H4 setting (6 min)


/home/nahshon/envs/ml001/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [16:36:53] WARNING: /workspace/src/common/error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  warnings.warn(smsg, UserWarning)


...

XGB H5: the H4 setting, hold-out RMSE 1.7730 dB against 1.7730 with the H4 setting (3 min)


XGB H6: the H4 setting, hold-out RMSE 1.7739 dB against 1.7739 with the H4 setting (4 min)


XGB H7: the H4 setting, hold-out RMSE 1.7730 dB against 1.7730 with the H4 setting (4 min)


the H4 setting  sample CV RMSE  hold-out RMSE, H4 setting  \
family rung                                                              
LGBM   H4              True          2.5682                     1.7696   
       H5              True          2.5651                     1.7655   
       H6             False          2.5686                     1.7670   
       H7             False          2.5605                     1.7672   
XGB    H4              True          2.5680                     1.7744   
       H5              True          2.5677                     1.7730   
       H6              True          2.5716                     1.7739   
       H7              True          2.5649                     1.7730   

             hold-out RMSE, own setting  difference      lo      hi  \
family rung                                                           
LGBM   H4                        1.7696      0.0000 -0.0000  0.0000   
       H5                        1.7655      0.0000 -0.0000  0.0000   
       H6                        1.7687      0.0017  0.0005  0.0030   
       H7                        1.7684      0.0012 -0.0005  0.0028   
XGB    H4                        1.7744      0.0000 -0.0000  0.0000   
       H5                        1.7730      0.0000 -0.0000  0.0000   
       H6                        1.7739      0.0000 -0.0000  0.0000   
       H7                        1.7730      0.0000 -0.0000  0.0000   

             margin 0.99, H4 setting  margin 0.99, own setting  \
family rung                                                      
LGBM   H4                     7.7305                    7.7305   
       H5                     7.7175                    7.7175   
       H6                     7.7645                    7.8070   
       H7                     7.7200                    7.8081   
XGB    H4                     7.8673                    7.8673   
       H5                     7.8782                    7.8782   
       H6                     7.8850                    7.8850   
       H7                     7.8548                    7.8548   

             reached 0.99, own setting  
family rung                             
LGBM   H4                       0.9985  
       H5                       0.9985  
       H6                       0.9985  
       H7                       0.9986  
XGB    H4                       0.9986  
       H5                       0.9986  
       H6                       0.9986  
       H7                       0.9986

setting
family rung                                                             
LGBM   H4    {"colsample_bytree": 0.9062, "learning_rate": 0.0167, "m...
       H5    {"colsample_bytree": 0.9062, "learning_rate": 0.0167, "m...
       H6    {"colsample_bytree": 0.5029, "learning_rate": 0.01, "min...
       H7    {"colsample_bytree": 0.5, "learning_rate": 0.0114, "min_...
XGB    H4    {"colsample_bytree": 0.8698, "learning_rate": 0.0249, "m...
       H5    {"colsample_bytree": 0.8698, "learning_rate": 0.0249, "m...
       H6    {"colsample_bytree": 0.8698, "learning_rate": 0.0249, "m...
       H7    {"colsample_bytree": 0.8698, "learning_rate": 0.0249, "m...

Reading of the results.

* Tuning on the rung's own inputs does not change the picture. XGBoost chooses the setting tuned on rung H4 on every rung from H4 to H7, so its results are those of notebook 13. LightGBM chooses that setting on H4 and H5 and other settings on H6 and H7, which do worse on the hold-out by 0.0017 dB (interval 0.0005 to 0.0030) and 0.0012 dB (interval -0.0005 to +0.0028); with them the 99 % margin is 7.81 dB, against 7.76 and 7.72 dB with the H4 setting, at a coverage of 99.85 to 99.86 %.
* The search on rung H4 returns the setting of notebook 13 for both families, and the refit returns its hold-out residuals.
* The SNR of earlier packets, the sensors and the level and geometry of the link add nothing to the history, whether the model is tuned on rung H4 or on their own inputs.